<a href="https://colab.research.google.com/github/VizDataWorks/eCommerce_data_analysis/blob/main/ecommerce-challenge/notebooks/00_data_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 00 – Data Preparation: Cleaning the Raw Files

**Capital One E-Commerce Data Challenge · Step 1 of 6**

This notebook downloads the three raw datasets, cleans each one, and saves the cleaned tables.

Every table follows the same order: **trim spaces → remove duplicates → fix types and values → validate.**

| Table | Raw | Clean | Output file |
| --- | --- | --- | --- |
| Customers (`buyer.csv`) | 52,500 rows | 47,500 rows | `data/clean/customers.csv` |
| Products (11 vendor files) | 1,037 rows | 1,037 rows | `data/clean/products.csv` |
| Sales (`sales.csv`) | 376,004 rows | 348,905 rows | `data/clean/sales.csv` |

**Project folder:** `ecommerce-challenge/` holds `data/` and `notebooks/`.

*   `data/raw/` has the original files
*   `data/clean/` stores the cleaned files
*   ecommerce_challenge/notebooks' holds Data Analysis notebooks


**Contents:** 1. Setup · 2. Raw files · 3. Helpers · 4. Customers · 5. Products · 6. Sales · 7. Validation · 8. Save clean files · 9. Clean field dictionary

# 1.Setup


In [4]:
import os, sys, glob, shutil, subprocess
import numpy as np
import pandas as pd
from IPython.display import display

# Get the project repo from GitHub, raw files included: no sign-in needed
REPO = "https://github.com/VizDataWorks/eCommerce_data_analysis.git"
subprocess.run(["rm", "-rf", "repo"], check=True)   # start fresh on every run
subprocess.run(["git", "clone", "-q", "--depth", "1", REPO, "repo"], check=True)

# Project folders inside the downloaded copy
BASE    = os.path.join("repo", "ecommerce-challenge")
RAW     = os.path.join(BASE, "data", "raw")
VENDORS = os.path.join(RAW, "vendors")
CLEAN   = os.path.join(BASE, "data", "clean")

COMPANY_TZ = "America/Chicago"   # company is in Missouri (Central time)

# Show more columns when displaying tables
pd.set_option("display.max_columns", 30, "display.width", 160, "display.max_colwidth", 60)
print("Project folder:", BASE)
!ls -lh {RAW} {VENDORS}

Project folder: repo/ecommerce-challenge
repo/ecommerce-challenge/data/raw:
total 14M
-rw-r--r-- 1 root root 6.7M Nov  3 02:41 buyer.csv
-rw-r--r-- 1 root root 107K Nov  3 02:41 metadata.pdf
-rw-r--r-- 1 root root 6.8M Nov  3 02:41 sales.csv.gz
drwxr-xr-x 2 root root 4.0K Nov  3 02:41 vendors

repo/ecommerce-challenge/data/raw/vendors:
total 112K
-rw-r--r-- 1 root root 7.5K Nov  3 02:41 CPSFP.csv
-rw-r--r-- 1 root root  11K Nov  3 02:41 EYMKE.csv
-rw-r--r-- 1 root root  12K Nov  3 02:41 JDTOJ.csv
-rw-r--r-- 1 root root 1.7K Nov  3 02:41 JSCZL.csv
-rw-r--r-- 1 root root 9.4K Nov  3 02:41 KWOXE.csv
-rw-r--r-- 1 root root 4.8K Nov  3 02:41 NKDLI.csv
-rw-r--r-- 1 root root 4.1K Nov  3 02:41 PQYYY.csv
-rw-r--r-- 1 root root 7.9K Nov  3 02:41 ROPIK.csv
-rw-r--r-- 1 root root 9.9K Nov  3 02:41 SOPAJ.csv
-rw-r--r-- 1 root root 9.8K Nov  3 02:41 TAAFL.csv
-rw-r--r-- 1 root root  16K Nov  3 02:41 WQEPS.csv


## 3. Helper functions

Shared by all three tables. trim removes stray spaces and turns blanks into missing values; parse_dates reads the mixed date formats in the raw files (text dates, Unix timestamps, and 1/1/70 as “no date”).

In [7]:
def trim(df):
    """Strip leading/trailing spaces from every column (all read as text); blank strings become NaN."""
    return df.apply(lambda c: c.str.strip().replace("", np.nan))

def parse_dates(s):
    """Parse the mixed date formats. 'm/d/yy H:MM', 'm/d/yy', 'dd-Mon-yy' and Unix seconds (e.g. 1716118560).'1/1/70' is a placeholder for "no date" and becomes NaT. """
    s = s.astype("string")
    out = pd.Series(pd.NaT, index=s.index, dtype="datetime64[ns]")

    # Unix timestamps: 9-10 digits, in seconds
    unix = s.str.fullmatch(r"\d{9,10}(\.0)?", na=False)
    out[unix] = pd.to_datetime(s[unix].astype(float), unit="s")

    # Try each text format in turn, only on values not parsed yet
    for fmt in ["%m/%d/%y %H:%M", "%m/%d/%y", "%d-%b-%y"]:
        todo = ~unix & s.notna() & out.isna()
        out[todo] = pd.to_datetime(s[todo], format=fmt, errors="coerce")

    # '1/1/70' means "no date"
    out[s.str.startswith("1/1/70", na=False)] = pd.NaT
    return out

    #    """Convert buyer-local times to company time (Central)."""
    def to_company_time(local, tz):
      out = pd.Series(pd.NaT, index=local.index, dtype="datetime64[ns]")
      for zone in tz.dropna().unique():
        m = (tz == zone) & local.notna()
        # Read the time in the buyer's time zone, then convert to company time
        out[m] = (local[m].dt.tz_localize(zone, ambiguous="NaT", nonexistent="NaT")
                          .dt.tz_convert(COMPANY_TZ).dt.tz_localize(None))
    return out

## 4. Customers (`buyer.csv`): 52,500 rows → 47,500

*   Spaces trimmed first: 2,500 rows. Example - `" B123"` and
`"B123"` were merged be the same buyer.
*   2,500 exact duplicate rows removed.
*   Duplicate buyer_ids checked: the raw file has 4,999. After the
exact copies are removed, every remaining duplicate has a blank ID, so no real customer records needed merging
*   2,500 rows with a blank buyer_id dropped, because they can never be joined to an order.
*   Signup dates modified into real dates.
*   Blank region (4,763) and timezone (2,369) filled from the state. The notebook first confirms that each state has only one region and one timezone, so the fill is exact.
*  `is_referred` = 2 (473 rows) set to unknown rather than guessed, so it doesn't bias the referral analysis.



In [8]:
def load_customers():
    df = trim(pd.read_csv(os.path.join(RAW, "buyer.csv"), dtype=str)).drop_duplicates()

    # No buyer_id means the customer can't be joined to orders
    df = df.dropna(subset=["buyer_id"]).copy()
    df["signup_date"] = parse_dates(df.signup_date).dt.normalize()

    # Fill blank region/timezone from state (safe only if each state has one value)
    for col in ["region", "timezone"]:
        by_state = df.groupby("state")[col]
        assert by_state.nunique().max() == 1, f"a state has more than one {col}"
        df[col] = df[col].fillna(df.state.map(by_state.first()))

    # is_referred = 2 isn't valid: set to unknown rather than guess
    df["is_referred"] = pd.to_numeric(df.is_referred).where(lambda x: x.isin([0, 1])).astype("Int64")
    for c in ["is_active_buyer", "wishlist_size"]:
        df[c] = pd.to_numeric(df[c]).astype("Int64")

    assert df.buyer_id.is_unique
    return df.sort_values("buyer_id").reset_index(drop=True)


customers = load_customers()
display(customers.head())

,buyer_id,customer_group,is_active_buyer,signup_date,customer_segment,preferred_subcategory,subcategory_pool,preferred_channel,preferred_payment,region,state,timezone,is_referred,wishlist_size
0,B10000,VIP,1,2018-09-08,Fashion Shopper,Skirts,"Skirts, Jeans",App,Debit Card,West,NV,America/Los_Angeles,0,28
1,B10001,VIP,1,2019-08-20,Tech Enthusiast,Bluetooth Speakers,"Power Banks, Smartphones",Website,Credit Card,West,NV,America/Los_Angeles,0,9
2,B10002,VIP,1,2018-08-22,Fitness Buff,Yoga Mats,"Water Bottles, Yoga Mats",App,Digital Wallet,East,NJ,America/New_York,0,2
3,B10003,VIP,1,2023-04-23,Cozy Homemaker,Table Lamps,"Throw Pillows, Wall Art",Mobile,Digital Wallet,Central,TX,America/Chicago,0,11
4,B10004,VIP,1,2023-10-17,Fashion Shopper,Dresses,"Handbags, Skirts",Website,Digital Wallet,Central,WI,America/Chicago,1,29


## 5. Products (11 vendor files): 1,037 rows
*   Column names made consistent before the union: two files call the margin column `pct_profit_margin`.
*   Margins converted: JDTOJ and ROPIK store margin as a percentage, the other files as a fraction. The median margin per file shows the 100× gap.
*   IDs uppercased.
*   `$` and commas removed from prices (46 rows).
*  True/false flags with mixed spellings made consistent (151 rows).
*   New product key `sku_id`: 245 product numbers are sold by more than one vendor, so the key combines subcategory, vendor and product number.



In [12]:
FLAG_MAP = {"true": True, "y": True, "1": True, "false": False, "n": False, "0": False}


def load_products():
    files = sorted(glob.glob(os.path.join(VENDORS, "*.csv")))
    df = pd.concat([pd.read_csv(f, dtype=str)
                      .rename(columns={"pct_profit_margin": "profit_margin"})
                      .assign(source_file=os.path.basename(f))
                    for f in files], ignore_index=True)
    df = trim(df)

    # Some vendors send margin as a percent (25), others as a fraction (0.25).
    # A file whose median margin is above 1 is in percent, so divide it by 100.
    # Negative margins are kept: those products sell at a loss.
    df["profit_margin"] = pd.to_numeric(df.profit_margin.str.rstrip("%"))
    file_median = df.groupby("source_file").profit_margin.transform("median")
    df.loc[file_median > 1, "profit_margin"] /= 100

    id_cols = ["category_id", "subcategory_id", "vendor", "product_num"]
    df[id_cols] = df[id_cols].apply(lambda c: c.str.upper())
    df["price"] = pd.to_numeric(df.price.str.replace(r"[$,]", "", regex=True))
    for c in ["currently_active_vendor", "active_product"]:
        df[c] = df[c].str.lower().map(FLAG_MAP).astype("boolean")
    df["current_stock"] = pd.to_numeric(df.current_stock).astype("Int64")

    # product_num isn't unique (vendors reuse numbers), so the key adds subcategory and vendor
    df.insert(0, "sku_id", df.subcategory_id + "-" + df.vendor + "-" + df.product_num)
    assert df.sku_id.is_unique
    return df


products = load_products()
display(products.head())

,sku_id,category_name,category_id,subcategory_name,subcategory_id,vendor,currently_active_vendor,product_num,price,profit_margin,active_product,current_stock,source_file
0,PBESK-CPSFP-9DC69ECE1503,Beauty,PBE,Skincare,PBESK,CPSFP,True,9DC69ECE1503,9.91,0.767131,True,90,CPSFP.csv
1,PBESK-CPSFP-EF523B4FA026,Beauty,PBE,Skincare,PBESK,CPSFP,True,EF523B4FA026,13.42,0.767131,True,65,CPSFP.csv
2,PBESK-CPSFP-C541AC8EF917,Beauty,PBE,Skincare,PBESK,CPSFP,True,C541AC8EF917,60.74,0.767131,True,29,CPSFP.csv
3,PBESK-CPSFP-7CCFBF68193B,Beauty,PBE,Skincare,PBESK,CPSFP,True,7CCFBF68193B,55.82,0.767131,True,91,CPSFP.csv
4,PBESK-CPSFP-97FBE210389E,Beauty,PBE,Skincare,PBESK,CPSFP,True,97FBE210389E,55.17,0.767131,True,17,CPSFP.csv


## 6. Sales (`sales.csv`): 376,004 rows → 348,905


*   Rows removed: 24,878 exact duplicates.
*   2,221 repeated order+product lines collapsed into one. Each keeps its most common quantity, and on a tied status it is marked Cancelled, so revenue is never overstated.
*   Values fixed: 4,429 dates stored as Unix timestamps converted. 179 `1/1/70` placeholder dates set to null.
*   3,545 `Deliverred` typos corrected.
*   Rows kept but flagged:
o	394 bad quantities (missing or outside 1–3) are set to null and flagged.
o	1,662 orders with no status are labelled Unknown and left out of revenue.
o	16,889 lines from buyers missing from the customer table are flagged.
o	159 lines with no buyer_id at all are flagged too.


In [14]:
def to_company_time(local, tz):
    # Order times are in the buyer's local time zone; convert each zone to Central
    out = pd.Series(pd.NaT, index=local.index, dtype="datetime64[ns]")
    for zone in tz.dropna().unique():
        m = (tz == zone) & local.notna()
        out[m] = (local[m].dt.tz_localize(zone, ambiguous="NaT", nonexistent="NaT")
                          .dt.tz_convert(COMPANY_TZ).dt.tz_localize(None))
    return out


def most_common_status(s):
    top = s.mode()
    if len(top) > 1:
        return "Cancelled"   # tie: don't count it as revenue
    return top.iloc[0] if len(top) else pd.NA


def load_sales(customers):
    df = trim(pd.read_csv(os.path.join(RAW, "sales.csv.gz"), dtype=str)).drop_duplicates()

    # Date-only values like '6/26/24' have no real time of day, so they get no company time
    has_time = (df.order_datetime.str.contains(":", na=False)
                | df.order_datetime.str.fullmatch(r"\d{9,10}(\.0)?", na=False))
    df["order_datetime"] = parse_dates(df.order_datetime)
    df["order_status"] = df.order_status.replace({"Deliverred": "Delivered"})

    # Valid quantities are 1-3; anything else becomes blank and is flagged below
    q = pd.to_numeric(df.quantity, errors="coerce")
    df["quantity"] = q.where(q.between(1, 3)).astype("Int64")

    # Fill a blank buyer_id or status from the same order, if that order has only one value
    for col in ["buyer_id", "order_status"]:
        per_order = df.groupby("order_id")[col]
        single = per_order.first()[per_order.nunique() == 1]
        df[col] = df[col].fillna(df.order_id.map(single))

    tz = df.buyer_id.map(customers.set_index("buyer_id").timezone)
    df["order_datetime_company"] = to_company_time(df.order_datetime.where(has_time), tz)
    # Some orders mix timed and date-only lines: give the whole order one time
    df["order_datetime_company"] = df.groupby("order_id").order_datetime_company.transform("min")

    # Same order + product on several lines: keep one line, with the most common
    # quantity (tie -> smaller) and status (tie -> Cancelled, so revenue isn't overstated)
    key = ["order_id", "sku_id"]
    dup = df.duplicated(key, keep=False)
    picked = df[dup].groupby(key).agg(quantity=("quantity", lambda q: q.mode().min()),
                                      order_status=("order_status", most_common_status))
    merged = df[dup].drop_duplicates(key).drop(columns=["quantity", "order_status"]).join(picked, on=key)
    df = pd.concat([df[~dup], merged], ignore_index=True)
    df["quantity"] = df.quantity.astype("Int64")

    # Rows are kept but flagged; only Delivered orders count as revenue
    df["flag_bad_quantity"] = df.quantity.isna()
    df["order_status"] = df.order_status.fillna("Unknown")
    df["is_revenue"] = df.order_status.eq("Delivered")
    df["flag_missing_buyer_id"] = df.buyer_id.isna()
    df["flag_buyer_not_in_customers"] = df.buyer_id.notna() & ~df.buyer_id.isin(customers.buyer_id)

    return df.sort_values(["order_datetime", "order_id", "sku_id"]).reset_index(drop=True)


sales = load_sales(customers)
display(sales.head())

,order_id,buyer_id,sku_id,quantity,order_datetime,order_status,order_datetime_company,flag_bad_quantity,is_revenue,flag_missing_buyer_id,flag_buyer_not_in_customers
0,O00000003,B13344,PBEMA-WQEPS-0F47F542BAD6,1,2024-01-01 00:00:00,Delivered,NaT,False,True,False,False
1,O00000024,B15811,PBEMA-CPSFP-0F47F542BAD6,3,2024-01-01 00:00:00,Delivered,NaT,False,True,False,False
2,O00000043,B12027,PCLJA-WQEPS-F7551344E755,1,2024-01-01 00:00:00,Delivered,2024-01-01 16:42:00,False,True,False,False
3,O00000051,B12885,PCLJA-CPSFP-935B98D0317C,1,2024-01-01 00:55:00,Delivered,2024-01-01 02:55:00,False,True,False,False
4,O00000022,B10791,PLESP-SOPAJ-2D31003DCCCA,1,2024-01-01 02:54:00,Delivered,2024-01-01 02:54:00,False,True,False,False


## 7. Validation

These checks must all pass before the clean files are saved. They confirm the keys are unique and that every sales line can join to a product, so later joins cannot add or lose rows.

In [15]:
checks = {
    "customers: buyer_id unique":                   customers.buyer_id.is_unique,
    "customers: no blank region or timezone":       customers[["region", "timezone"]].notna().all().all(),
    "products: sku_id unique":                      products.sku_id.is_unique,
    "products: margins are fractions (-1 to 1)":    products.profit_margin.between(-1, 1).all(),
    "products: prices numeric and positive":        (products.price > 0).all(),
    "sales: one line per order + sku":              not sales.duplicated(["order_id", "sku_id"]).any(),
    "sales: every sku_id found in products":        sales.sku_id.isin(products.sku_id).all(),
    "sales: quantities are 1-3 or null":            sales.quantity.dropna().between(1, 3).all(),
    "sales: statuses are Delivered/Cancelled/Unknown": set(sales.order_status) <= {"Delivered", "Cancelled", "Unknown"},
    "sales: dates within 2024-2025":                sales.order_datetime.dropna().dt.year.between(2024, 2025).all(),
    "sales: one buyer per order":                   (sales.dropna(subset=["buyer_id"]).groupby("order_id").buyer_id.nunique() == 1).all(),
}
result = pd.DataFrame({"check": list(checks), "passed": [bool(v) for v in checks.values()]})
display(result)
assert result.passed.all(), "a validation check failed"

summary = pd.DataFrame({
    "table": ["customers", "products", "sales"],
    "clean_rows": [len(customers), len(products), len(sales)],
    "columns": [customers.shape[1], products.shape[1], sales.shape[1]],
})
display(summary)
print("Sales status mix:", sales.order_status.value_counts().to_dict())
print("Distinct orders:", f"{sales.order_id.nunique():,}",
      "| date range:", sales.order_datetime.min().date(), "to", sales.order_datetime.max().date())

,check,passed
0,customers: buyer_id unique,True
1,customers: no blank region or timezone,True
2,products: sku_id unique,True
3,products: margins are fractions (-1 to 1),True
4,products: prices numeric and positive,True
5,sales: one line per order + sku,True
6,sales: every sku_id found in products,True
7,sales: quantities are 1-3 or null,True
8,sales: statuses are Delivered/Cancelled/Unknown,True
9,sales: dates within 2024-2025,True


,table,clean_rows,columns
0,customers,47500,14
1,products,1037,13
2,sales,348905,11


Sales status mix: {'Delivered': 336730, 'Cancelled': 10513, 'Unknown': 1662}
Distinct orders: 244,280 | date range: 2024-01-01 to 2025-12-30


## 8. Save the clean files

Written to `data/clean/`. Notebooks 01–05 read these files instead of repeating the cleaning.

In [17]:
customers.to_csv(os.path.join(CLEAN, "customers.csv"), index=False)
products.to_csv(os.path.join(CLEAN, "products.csv"), index=False)
sales.to_csv(os.path.join(CLEAN, "sales.csv"), index=False)
show_tree(os.path.join(BASE, "data"))

data/
    clean/
        customers.csv  (6.29 MB)
        data_quality_log.csv  (0.00 MB)
        products.csv  (0.13 MB)
        sales.csv  (40.40 MB)
        sales.csv.gz  (6.05 MB)
    raw/
        buyer.csv  (6.97 MB)
        metadata.pdf  (0.11 MB)
        sales.csv.gz  (7.03 MB)
        vendors/
            CPSFP.csv  (0.01 MB)
            EYMKE.csv  (0.01 MB)
            JDTOJ.csv  (0.01 MB)
            JSCZL.csv  (0.00 MB)
            KWOXE.csv  (0.01 MB)
            NKDLI.csv  (0.00 MB)
            PQYYY.csv  (0.00 MB)
            ROPIK.csv  (0.01 MB)
            SOPAJ.csv  (0.01 MB)
            TAAFL.csv  (0.01 MB)
            WQEPS.csv  (0.02 MB)
